# Asesor de Matrícula: Demo del Agente

Agente que asesora a un estudiante sobre su matrícula consultando el **pensum** y su
**historial de notas** mediante herramientas. Usa el agentic loop de *part01* con el
presupuesto de tool calls de *part03*, sin frameworks.

| Herramienta | Qué hace |
|---|---|
| `buscar_materia` | Busca una materia por código o nombre |
| `verificar_prerrequisitos` | Revisa si cumple los prerrequisitos |
| `materias_disponibles` | Lista lo que puede inscribir (opcional: por semestre) |
| `resumen_academico` | Créditos, promedio, materias perdidas |
| `validar_carga` | Valida un conjunto de materias (máx. 21 créditos) |

---
## ⚙️ Sección 1: Setup

**Antes de ejecutar:** en Colab, abre el panel 🔑 **Secretos** (barra izquierda), crea el
secreto `NVIDIA_API_KEY` con tu clave (empieza con `nvapi-`) y activa el acceso para este
notebook. La clave **nunca** se escribe en el código.

In [ ]:
%pip install openai python-dotenv --quiet

In [ ]:
# Descarga el proyecto (agente.py, herramientas) desde GitHub.
# Si ejecutas este notebook localmente desde la carpeta del repo, omite esta celda.
import os

REPO = "https://github.com/laurasjparis/asesor-matricula-agente.git"

if not os.path.exists("asesor-matricula-agente"):
    !git clone {REPO}
%cd asesor-matricula-agente

In [ ]:
from agente import run_agent
import json

def mostrar_traza(traza):
    """Imprime cada tool call: herramienta, argumentos y resultado."""
    print(f"\n{'═'*60}\nTRAZA: {len(traza)} tool calls\n{'═'*60}")
    for i, paso in enumerate(traza, 1):
        print(f"\n[{i}] 🔧 {paso['herramienta']}")
        print(f"    argumentos: {json.dumps(paso['argumentos'], ensure_ascii=False)}")
        print(f"    resultado:  {json.dumps(paso['resultado'], ensure_ascii=False, indent=2)}")

---
## 🧪 Sección 2: Pregunta 1, consulta normal

El estudiante pregunta por materias **por su nombre**. El agente debe buscar los códigos,
validar la carga y responder solo con lo que devuelven las herramientas.

In [ ]:
respuesta, traza = run_agent(
    "Quiero inscribir Física Mecánica, Estructuras de Datos y Probabilidad "
    "el próximo semestre. ¿Puedo? ¿Cuántos créditos serían?",
    max_tool_calls=6,
)
mostrar_traza(traza)

---
## 🧪 Sección 3: Pregunta 2, presupuesto agotado

La pregunta necesita varias herramientas, pero el presupuesto es de **1 tool call**.
Al agotarse, el agente responde a las llamadas pendientes con `"presupuesto agotado"` y hace
una última llamada al modelo **sin tools**, que debe indicar qué no pudo verificar.

In [ ]:
respuesta, traza = run_agent(
    "Dame mi promedio, dime qué materias puedo ver en tercer semestre "
    "y si ya puedo inscribir Introducción a la IA.",
    max_tool_calls=1,
)
mostrar_traza(traza)

---
## 🚀 Prueba otras preguntas

- Una materia que no existe: *"¿Puedo inscribir Termodinámica?"*
- Una consulta ambigua: *"¿Puedo ver Cálculo?"* (hay Cálculo I, II y III)
- Exceso de créditos: pide inscribir 6 o más materias a la vez

In [ ]:
respuesta, traza = run_agent("...")
mostrar_traza(traza)